# fusdb_pyomo Examples

This notebook walks through the main `fusdb_pyomo` workflow: registry loading, reactor inspection, temporary scenario edits, and the four execution styles used by `ARC_2015`.

The solve cells assume an NLP solver such as `ipopt` is available.

In [ ]:
from pathlib import Path
import sys
import warnings

root = Path.cwd().resolve()
while root != root.parent and not (root / "src" / "fusdb").is_dir():
    root = root.parent

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

warnings.filterwarnings("ignore", message="Multiple registered relations define outputs.*")

from fusdb import Reactor, Variable
from fusdb.registry import RELATIONS, VARIABLES


def load_arc():
    return Reactor.from_yaml(root / "reactors" / "ARC_2015" / "reactor.yaml")

In [2]:
p_fus_spec = VARIABLES.get("P_fus")
fusion_power_relation = RELATIONS.get_filtered_relations(names=("Total fusion power",))[0]

print("Loaded variable registry entry:")
print("  name:", p_fus_spec.name)
print("  unit:", p_fus_spec.unit)
print("  shape:", p_fus_spec.shape)

print("Loaded relation registry entry:")
print("  name:", fusion_power_relation.name)
print("  tags:", fusion_power_relation.tags)
print("  output_names:", fusion_power_relation.output_names)

Loaded variable registry entry:
  name: P_fus
  unit: W
  shape: 0
Loaded relation registry entry:
  name: Total auxiliary power
  tags: ('power_exhaust', 'auxiliary')
  output_names: ('P_aux',)


In [3]:
demo_variable = Variable("P_fus", value=525, unit="MW")
demo_relation = RELATIONS.get_filtered_relations(names=("Total fusion power",))[0]

print("Variable properties:")
print("  name:", demo_variable.name)
print("  unit:", demo_variable.unit)
print("  shape:", demo_variable.shape)
print("  value:", demo_variable.value)
print("  fixed:", demo_variable.fixed)
print("  rel_tol:", demo_variable.rel_tol)

print("Relation properties:")
print("  name:", demo_relation.name)
print("  tags:", demo_relation.tags)
print("  input_names:", demo_relation.input_names)
print("  output_names:", demo_relation.output_names)
print("  variables:", demo_relation.variables)

Variable properties:
  name: P_fus
  unit: W
  shape: 0
  value: 525000000.0
  fixed: False
  rel_tol: 0.01
Relation properties:
  name: Total auxiliary power
  tags: ('power_exhaust', 'auxiliary')
  input_names: ('P_NBI', 'P_ICRF', 'P_LHCD')
  output_names: ('P_aux',)
  variables: ('P_NBI', 'P_ICRF', 'P_LHCD', 'P_aux')


In [4]:
arc = load_arc()

print(arc)
print("Loaded variables:", len(arc.variables))
print("Loaded mode:", arc.mode)

Reactor(name='ARC 2015', organization='MIT', country='USA', year=2015, doi='10.1016/j.fusengdes.2015.07.008', notes='Conceptual compact tokamak from MIT', tags=('tokamak', 'compact_tokamak', 'i_mode'), mode='verify', variables={'R': Variable(name='R', value=3.3, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=3.3, relations=()), 'a': Variable(name='a', value=1.13, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=1.13, relations=()), 'A': Variable(name='A', value=2.92, unit='dimensionless', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=2.92, relations=()), 'P_fus': Variable(name='P_fus', value=525000000.0, unit='W', rel_tol=0.01, fixed=False, size=None, constraints=None, shape=0, source='given', freedom='fr

In [5]:
print("ARC_2015 variables through attribute access:")
print("  R:", arc.R.value)
print("  a:", arc.a.value)
print("  P_fus:", arc.P_fus.value)
print("  T_e length:", len(arc.T_e.value))
print("  T_e first three values:", arc.T_e.value[:3])

ARC_2015 variables through attribute access:
  R: 3.3
  a: 1.13
  P_fus: 525000000.0
  T_e length: 46
  T_e first three values: [26.74411184 26.46336996 25.94679197]


In [6]:
temp_variable = Variable("P_aux", value=50, unit="MW", constraints=("P_aux <= 75e6",))
arc.add_variable(temp_variable)

temp_system = arc.to_relation_system()
temp_relation_names = [relation.name for relation in temp_system.relations if relation.source_name == "P_aux"]

print("Temporary variable value in watts:", arc.P_aux.value)
print("Temporary relations attached to P_aux:", temp_relation_names)

removed_variable = arc.variables.pop("P_aux")
print("Removed variable:", removed_variable.name)
print("P_aux still present after cleanup:", "P_aux" in arc.variables)

Temporary variable value in watts: 50000000.0
Temporary relations attached to P_aux: ['P_aux_constraint_0']
Removed variable: P_aux
P_aux still present after cleanup: False


In [7]:
arc = load_arc()
arc.mode = "verify"
verify_result = arc.run()
print("Verification result:", verify_result)

Verification result: {'mode': 'verify', 'success': False, 'variables': {'R': Variable(name='R', value=3.3, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='consistent', reference_value=3.3, relations=()), 'a': Variable(name='a', value=1.13, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='fixed_inconsistent', reference_value=1.13, relations=()), 'A': Variable(name='A', value=2.92, unit='dimensionless', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='consistent', reference_value=2.92, relations=()), 'P_fus': Variable(name='P_fus', value=525000000.0, unit='W', rel_tol=0.01, fixed=False, size=None, constraints=None, shape=0, source='given', freedom='free', validity='unused', reference_value=525000000.0, relations=()), 'kappa': Variable(name='kappa', value=1.84, unit='dimensionless', rel_tol=0.01, 

In [8]:
arc = load_arc()
arc.mode = "reconcile"
reconcile_result = arc.run()

print("Reconciliation result:", reconcile_result)

Reconciliation result: {'mode': 'reconcile', 'success': True, 'variables': {'R': Variable(name='R', value=3.3, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='consistent', reference_value=3.3, relations=()), 'a': Variable(name='a', value=1.13, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='consistent', reference_value=1.13, relations=()), 'A': Variable(name='A', value=2.92, unit='dimensionless', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='consistent', reference_value=2.92, relations=()), 'P_fus': Variable(name='P_fus', value=525000000.0, unit='W', rel_tol=0.01, fixed=False, size=None, constraints=None, shape=0, source='given', freedom='free', validity='unused', reference_value=525000000.0, relations=()), 'kappa': Variable(name='kappa', value=1.84, unit='dimensionless', rel_tol=0.01, fixe

In [9]:
arc = load_arc()
arc.mode = "optimize"
optimize_result = arc.run(
    objective=lambda pyo_vars: pyo_vars["P_fus"],
    sense="maximize",
    solver="ipopt",
    tee=False,
)
print("Optimization result:", optimize_result)

TypeError: run() got an unexpected keyword argument 'solver'

In [ ]:
arc = load_arc()
ordered_result = arc.to_relation_system().ordered(passes=1)
print("Ordered relation system result:", ordered_result)

Ordered relation system result: {'mode': 'ordered', 'success': False, 'variables': {'R': Variable(name='R', value=3.3, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=3.3, relations=()), 'a': Variable(name='a', value=1.13, unit='m', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=1.13, relations=()), 'A': Variable(name='A', value=2.92, unit='dimensionless', rel_tol=0.01, fixed=True, size=None, constraints=None, shape=0, source='given', freedom='fixed', validity='unresolved', reference_value=2.92, relations=()), 'P_fus': Variable(name='P_fus', value=525000000.0, unit='W', rel_tol=0.01, fixed=False, size=None, constraints=None, shape=0, source='given', freedom='free', validity='unresolved', reference_value=525000000.0, relations=()), 'kappa': Variable(name='kappa', value=1.84, unit='dimensionless', rel_to